# Test the selected driver-distraction model

This notebook tests the **already trained ResNet-18** on the fixed State Farm test split: **4,831 images from five drivers**. It does not train a model or download data. The selected checkpoint and input hashes are fixed before test scoring.

**Run locally:** in VS Code choose **Select Kernel → Python Environments → this project's `.venv`**. Disconnect the Kaggle remote kernel for this notebook. Your local CPU can perform this test; keep the existing training notebook and backups.

The test measures five distraction classes only. Eye closure, yawning, nodding, gaze and DMD are not evaluated here. This fixed benchmark was previously scored for MobileNet; test scores must not be used to select further models or recipes.


## 1. Choose inspection or testing

First leave `RUN_TESTING = False` and click **Run All**. Step 2 should say **READY**. Then change it to `True` and click **Run All** once. Keep `DEVICE = "cpu"` for the local environment.


In [ ]:
from pathlib import Path
import json
import sys

RUN_TESTING = False
DEVICE = "cpu"
BATCH_SIZE = 32

assert not Path("/kaggle/working").is_dir(), (
    "Select the local project .venv Python kernel. This testing notebook uses your local files."
)
REPO = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "configs/selected_distraction_checkpoint.json").is_file()),
    None,
)
assert REPO is not None, "Open this notebook from inside the project folder."
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
OUTPUT = REPO / "results/testing/state_farm_resnet18_frozen"
print("Python:", sys.executable)
print("Project:", REPO)
print("Testing enabled:", RUN_TESTING, "| Device:", DEVICE)
print("Results:", OUTPUT)


## 2. Verify the saved model and test data

This checks the checkpoint's SHA-256, model configuration, validation selection, original manifest and exact driver assignments. It checks test paths without decoding images or making predictions. Missing files or changed hashes stop the run.


In [ ]:
from src.eval.frozen_distraction import inspect_inputs

plan = inspect_inputs(REPO)
print("READY:", plan["ready"])
print("Selected model:", plan["selection"]["model"])
print("Selected epoch:", plan["selection"]["best_epoch"])
print("Selected using:", plan["selection"]["selection_metric"])
print("Fixed test images:", plan["test_samples"])
print("Fixed test drivers:", plan["test_subjects"])
print("Checkpoint:", plan["model_dir"])
print("Training performed:", plan["training_performed"])


## 3. Run the fixed test

When enabled, this loads the exact saved weights, uses evaluation preprocessing, and predicts **only test images**. It prints progress after the first batch and every 20 batches. CPU inference can take several minutes; let it finish.

Results are saved locally as JSON and CSV. A completed result is reused on a repeated Run All. An incomplete/nonempty output folder raises an error rather than overwriting it.


In [ ]:
from src.eval.frozen_distraction import evaluate

if not RUN_TESTING:
    print("Inspection only. No inference or training performed.")
elif (OUTPUT / "summary.json").is_file():
    existing = json.loads((OUTPUT / "summary.json").read_text(encoding="utf-8"))
    assert existing["checkpoint_sha256"] == plan["selection"]["checkpoint_sha256"]
    assert existing["test_samples"] == plan["test_samples"]
    print("Completed test already exists. Preserving it; see Step 4.")
else:
    result = evaluate(REPO, OUTPUT, device=DEVICE, batch_size=BATCH_SIZE)
    print("TEST COMPLETE. Results saved locally:", OUTPUT)


## 4. Read the results

**Accuracy** is the fraction of correct frames. **Macro-F1** gives each class equal weight. **Recall** tells us how many actual frames of a class were detected. Check each class and driver as well as overall accuracy.

These are test results for the frozen model; do not change the selected model, class mapping or thresholds based on them. Validation accuracy (87.68%) is not a promise of the same test accuracy.


In [ ]:
from IPython.display import Markdown, display

if not (OUTPUT / "summary.json").is_file():
    print("No completed test result yet. Enable RUN_TESTING in Step 1 when READY.")
else:
    summary = json.loads((OUTPUT / "summary.json").read_text(encoding="utf-8"))
    report = json.loads((OUTPUT / "test.json").read_text(encoding="utf-8"))
    print(f"TEST accuracy: {summary['accuracy']:.2%}")
    print(f"TEST macro-F1: {summary['macro_f1']:.2%}")
    print(f"Images: {summary['test_samples']:,} | Drivers: {summary['test_subjects']}")
    print("Training performed:", summary["training_performed"])
    lines = ["| Class | Precision | Recall | F1 | Images |",
             "|---|---:|---:|---:|---:|"]
    for name in report["class_order"]:
        row = report["per_class"][name]
        lines.append(
            f"| {name} | {row['precision']:.2%} | {row['recall']:.2%} | "
            f"{row['f1-score']:.2%} | {int(row['support'])} |"
        )
    display(Markdown("\n".join(lines)))
    drivers = ["| Test driver | Accuracy | Images |", "|---|---:|---:|"]
    for subject, row in summary["per_driver"].items():
        drivers.append(f"| {subject} | {row['accuracy']:.2%} | {row['images']} |")
    display(Markdown("\n".join(drivers)))
    matrix = ["| Actual / predicted | " + " | ".join(report["class_order"]) + " |",
              "|---|" + "---:|" * len(report["class_order"])]
    for name, counts in zip(report["class_order"], report["confusion_matrix"], strict=True):
        matrix.append("| " + name + " | " + " | ".join(map(str, counts)) + " |")
    display(Markdown("\n".join(matrix)))


## 5. Save and share the results

The results already live on your computer. Press **Ctrl+S** to keep the displayed report in this notebook. The ZIP below contains only the small test reports and predictions; it does not copy raw data or model weights. Tell the coding agent the test is complete so it can verify the outputs and prepare video integration.


In [ ]:
import shutil

if (OUTPUT / "summary.json").is_file():
    archive = shutil.make_archive(str(OUTPUT) + "_reports", "zip", OUTPUT)
    print("Report ZIP:", archive)
    print("Files:")
    for path in sorted(OUTPUT.iterdir()):
        print(" ", path.name)
else:
    print("No test reports to package yet.")
